# 1.2 剪枝（Pruning）

## 什么是剪枝？

剪枝是通过移除模型中对输出贡献较小的参数（权重、神经元、注意力头等），直接减少模型参数量和计算量的技术。剪枝后的模型产生稀疏结构，非零参数减少，从而降低存储和计算需求。

## 为什么用剪枝？

1. **参数冗余**：研究表明深度神经网络存在大量冗余参数，很多权重接近于0，移除它们对精度影响极小。Lottery Ticket Hypothesis指出，稠密网络中存在一个稀疏的"中奖彩票"子网络，单独训练即可达到相当精度。
2. **直接减少计算量**：与量化不同，剪枝是真正移除参数，结构化剪枝后模型变小，无需特殊硬件即可加速。
3. **与量化互补**：剪枝+量化可以叠加使用，先剪枝移除冗余参数，再量化压缩剩余参数，获得更大压缩比。

## 剪枝的分类

根据剪枝粒度，可分为三类：
- **非结构化剪枝**：移除单个权重，粒度最细，精度损失最小，但需要稀疏计算支持
- **结构化剪枝**：移除整个结构单元（头、层、通道），无需稀疏硬件即可加速
- **半结构化剪枝**：如N:M稀疏，在固定稀疏模式下剪枝，兼顾精度和硬件加速

### 剪枝的数学形式化

剪枝可形式化为一个优化问题：
$$\min_{m} L(W \odot m) \quad \text{s.t.} \quad \|m\|_0 \leq k$$

其中：
- $W$：模型权重
- $m$：二值掩码（0/1），$m_i = 0$表示第$i$个权重被剪枝
- $\odot$：逐元素乘法
- $L$：损失函数
- $k$：允许保留的最大参数数量
- $\|m\|_0$：掩码中非零元素的数量（L0范数）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import time

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

---
## 1.2.1 非结构化剪枝（Unstructured Pruning）

### 什么是非结构化剪枝？

非结构化剪枝将单个权重置零，产生稀疏权重矩阵。剪枝粒度最细，理论上精度损失最小，但需要稀疏计算硬件/软件支持才能获得实际加速。

### 为什么用非结构化剪枝？

1. **精度最优**：逐权重剪枝可以精确移除最不重要的参数，保留最重要的参数，精度损失最小。
2. **灵活性高**：可以任意稀疏度剪枝，不受结构约束。
3. **理论保证**：Lottery Ticket Hypothesis表明，非结构化剪枝可以找到高性能的稀疏子网络。

### 局限性

非结构化剪枝产生的稀疏矩阵是不规则的，在通用硬件（CPU/GPU）上无法直接加速，因为内存访问仍然是稠密的。需要专门的稀疏计算库或硬件支持。

### 幅度剪枝（Magnitude Pruning）

#### 什么是幅度剪枝？

按权重绝对值大小排序，移除绝对值最小的权重。简单直接，是最经典的剪枝方法。

#### 为什么幅度剪枝有效？

直觉上，绝对值小的权重对输出的贡献也小，移除它们对模型行为的影响最小。

#### 幅度剪枝的数学公式

**重要性分数**：
$$I(w_i) = |w_i|$$

**剪枝掩码**：
$$m_i = \begin{cases} 1 & \text{if } |w_i| > \tau \\ 0 & \text{otherwise} \end{cases}$$

其中：
- $w_i$：第$i$个权重
- $I(w_i)$：权重的重要性分数
- $\tau$：剪枝阈值，由目标稀疏度决定
- $m_i$：二值掩码，1表示保留，0表示剪枝
- 阈值$\tau$的确定：将所有权重的绝对值排序，取第$k$小的值作为阈值，$k$为目标剪枝数量

In [ ]:
def magnitude_pruning(weight: torch.Tensor, sparsity: float = 0.5) -> tuple:
    """幅度剪枝：将绝对值最小的权重置零"""
    mask = torch.ones_like(weight)
    n_prune = int(sparsity * weight.numel())
    if n_prune == 0:
        return weight, mask
    threshold = weight.abs().flatten().kthvalue(n_prune).values
    mask = (weight.abs() > threshold).float()
    pruned_weight = weight * mask
    return pruned_weight, mask

weight = torch.randn(128, 256)

sparsities = [0.3, 0.5, 0.7, 0.9]
print(f"{'稀疏度':<10} {'非零参数':<15} {'MSE(原始vs剪枝)':<20} {'余弦相似度':<15}")
print("-" * 60)

for s in sparsities:
    pw, mask = magnitude_pruning(weight, sparsity=s)
    nonzero = mask.sum().item()
    mse = F.mse_loss(weight, pw)
    cos = F.cosine_similarity(weight.unsqueeze(0), pw.unsqueeze(0)).mean()
    print(f"{s:<10.1f} {nonzero:<15.0f} {mse:<20.6f} {cos:<15.6f}")

print(f"\n关键: 非结构化剪枝产生稀疏矩阵，需要稀疏计算支持才能加速")
print(f"实际存储: 稀疏矩阵仍需存储mask，压缩比受限")

### 梯度剪枝（Gradient-based Pruning）

#### 什么是梯度剪枝？

梯度剪枝基于梯度信息评估权重重要性，移除对损失函数影响最小的权重。相比幅度剪枝仅考虑权重本身的大小，梯度剪枝考虑了权重对损失函数的实际影响。

#### 为什么梯度剪枝比幅度剪枝更准确？

幅度剪枝假设权重绝对值小就不重要，但这不一定成立——一个小权重如果梯度很大，说明模型正在积极调整它，移除它可能影响很大。梯度剪枝综合考虑权重值和梯度，更准确地评估重要性。

#### 梯度剪枝的数学公式

使用Taylor展开一阶近似，移除权重$w_i$对损失的影响：

$$\Delta L_i = |L(w_i = 0) - L(w_i)| \approx |w_i \cdot g_i|$$

**重要性分数**：
$$I(w_i) = |w_i \cdot g_i|$$

其中：
- $w_i$：第$i$个权重的值
- $g_i = \frac{\partial L}{\partial w_i}$：损失函数对第$i$个权重的梯度
- $\Delta L_i$：移除第$i$个权重后损失的近似变化量
- 一阶Taylor近似假设：$L(0) \approx L(w_i) - w_i \cdot g_i$
- 重要性分数越大，该权重对损失影响越大，不应被剪枝

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 2)
        self.fc2 = nn.Linear(dim * 2, dim)
        self.fc3 = nn.Linear(dim, 10)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def gradient_based_pruning(model, dataloader, sparsity=0.5, n_batches=10):
    """基于梯度的剪枝：使用|w * grad|作为重要性分数"""
    importance_scores = {}
    for name, param in model.named_parameters():
        if 'weight' in name:
            importance_scores[name] = torch.zeros_like(param)

    model.eval()
    for i, (x, y) in enumerate(dataloader):
        if i >= n_batches:
            break
        model.zero_grad()
        output = model(x)
        loss = F.cross_entropy(output, y)
        loss.backward()
        for name, param in model.named_parameters():
            if name in importance_scores and param.grad is not None:
                importance_scores[name] += (param.data * param.grad).abs()

    masks = {}
    for name, scores in importance_scores.items():
        n_prune = int(sparsity * scores.numel())
        if n_prune == 0:
            masks[name] = torch.ones_like(scores)
            continue
        threshold = scores.flatten().kthvalue(n_prune).values
        masks[name] = (scores > threshold).float()

    return masks

def apply_pruning_masks(model, masks):
    """应用剪枝mask到模型"""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in masks:
                param.data *= masks[name]

dim = 64
model = SimpleMLP(dim)
x_data = torch.randn(256, dim)
y_data = torch.randint(0, 10, (256,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=32)

model_copy = copy.deepcopy(model)
with torch.no_grad():
    out_before = model(x_data)
    acc_before = (out_before.argmax(1) == y_data).float().mean()

masks = gradient_based_pruning(model, loader, sparsity=0.5, n_batches=10)
apply_pruning_masks(model, masks)

with torch.no_grad():
    out_after = model(x_data)
    acc_after = (out_after.argmax(1) == y_data).float().mean()

total_params = sum(p.numel() for p in model.parameters())
zero_params = sum((p == 0).sum().item() for p in model.parameters())
actual_sparsity = zero_params / total_params

print(f"=== 梯度剪枝效果 ===")
print(f"剪枝前准确率: {acc_before:.4f}")
print(f"剪枝后准确率: {acc_after:.4f}")
print(f"实际稀疏度: {actual_sparsity:.2%}")
print(f"\n各层稀疏度:")
for name, mask in masks.items():
    layer_sparsity = 1 - mask.mean().item()
    print(f"  {name}: {layer_sparsity:.2%}")

### 渐进式幅度剪枝（Gradual Magnitude Pruning, GMP）

#### 什么是GMP？

GMP在训练过程中从0%稀疏度逐步增长到目标稀疏度，让模型逐步适应稀疏结构，比一次性剪枝精度更高。

#### 为什么GMP比一次性剪枝更好？

1. **逐步适应**：一次性剪枝突然移除大量参数，模型没有机会调整剩余参数来补偿。GMP逐步剪枝，每一步剪枝后模型都有机会通过训练恢复精度。
2. **更优的稀疏结构**：渐进剪枝让模型在训练过程中探索不同的稀疏结构，最终找到更好的稀疏子网络。
3. **实践验证**：TensorFlow Model Optimization和NVIDIA的稀疏训练流程均采用GMP作为标准方案。

#### GMP的数学公式

稀疏度调度（cubic schedule）：
$$s(t) = s_f + (s_i - s_f) \left(1 - \frac{t - t_0}{\Delta t}\right)^3$$

其中：
- $s(t)$：第$t$步的稀疏度
- $s_i$：初始稀疏度（通常为0）
- $s_f$：目标稀疏度（如0.8）
- $t_0$：开始剪枝的训练步数
- $\Delta t$：剪枝持续的步数
- 立方调度使剪枝速度先慢后快再慢，让模型在关键阶段有更多时间适应

In [ ]:
def gradual_magnitude_pruning_train(model, train_loader, target_sparsity=0.8,
                                    total_epochs=50, prune_start_epoch=5, prune_end_epoch=40):
    """渐进式幅度剪枝训练"""
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    masks = {name: torch.ones_like(p) for name, p in model.named_parameters() if 'weight' in name}
    sparsity_history = []
    loss_history = []

    for epoch in range(total_epochs):
        if prune_start_epoch <= epoch < prune_end_epoch:
            progress = (epoch - prune_start_epoch) / (prune_end_epoch - prune_start_epoch)
            current_sparsity = target_sparsity * progress
        elif epoch >= prune_end_epoch:
            current_sparsity = target_sparsity
        else:
            current_sparsity = 0.0

        for name, param in model.named_parameters():
            if name in masks:
                n_prune = int(current_sparsity * param.numel())
                if n_prune > 0:
                    threshold = param.data.abs().flatten().kthvalue(n_prune).values
                    masks[name] = (param.data.abs() > threshold).float()
                    param.data *= masks[name]

        epoch_loss = 0
        n_batches = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            output = model(x)
            loss = F.cross_entropy(output, y)
            loss.backward()
            optimizer.step()
            for name, param in model.named_parameters():
                if name in masks:
                    param.data *= masks[name]
            epoch_loss += loss.item()
            n_batches += 1

        sparsity_history.append(current_sparsity)
        loss_history.append(epoch_loss / n_batches)

    return masks, sparsity_history, loss_history

model_gmp = SimpleMLP(dim=64)
masks_gmp, sparsity_hist, loss_hist = gradual_magnitude_pruning_train(
    model_gmp, loader, target_sparsity=0.7, total_epochs=30,
    prune_start_epoch=3, prune_end_epoch=25
)

total_params = sum(p.numel() for p in model_gmp.parameters())
zero_params = sum((p == 0).sum().item() for p in model_gmp.parameters())

print(f"=== 渐进式幅度剪枝 (GMP) ===")
print(f"最终稀疏度: {zero_params/total_params:.2%}")
print(f"训练损失变化: {loss_hist[0]:.4f} -> {loss_hist[-1]:.4f}")
print(f"稀疏度变化: {sparsity_hist[0]:.2%} -> {sparsity_hist[-1]:.2%}")
print(f"\nGMP核心优势: 模型逐步适应稀疏结构，比一次性剪枝精度更高")

---
## 1.2.2 结构化剪枝（Structured Pruning）

### 什么是结构化剪枝？

结构化剪枝按结构化单元（注意力头、FFN中间维度、整层等）进行剪枝，剪枝后模型仍是稠密矩阵，无需稀疏计算支持即可获得实际加速。

### 为什么用结构化剪枝？

1. **实际加速**：剪枝后移除整个结构单元，模型变小，在任意硬件上都能加速，无需稀疏计算支持。
2. **内存友好**：模型参数真正减少，内存占用和带宽需求直接降低。
3. **部署简单**：剪枝后就是一个小模型，可以直接用现有推理框架部署，无需特殊处理。

### 结构化剪枝的数学原理

结构化剪枝可以表示为移除权重矩阵的整行或整列：

$$W' = W[\mathcal{S}_{out}, \mathcal{S}_{in}]$$

其中：
- $W \in \mathbb{R}^{m \times n}$：原始权重矩阵
- $\mathcal{S}_{out}$：保留的输出通道索引集合
- $\mathcal{S}_{in}$：保留的输入通道索引集合
- $W'$：剪枝后的权重矩阵，维度为$|\mathcal{S}_{out}| \times |\mathcal{S}_{in}|$

### 注意力头剪枝（Attention Head Pruning）

#### 什么是注意力头剪枝？

多头注意力中，不同头可能学习到冗余的模式。注意力头剪枝识别并移除不重要的头，减少注意力计算量。

#### 重要性评估

注意力头的重要性可通过其输出的L2范数来衡量：
$$I(h) = \sum_{x \in \mathcal{D}} \|\text{Head}_h(x)\|_2$$

其中 $\text{Head}_h(x)$ 是第$h$个头对输入$x$的输出，$\mathcal{D}$是校准数据集。

In [ ]:
class MultiHeadAttentionWithPruning(nn.Module):
    """支持头剪枝的多头注意力
    注意：本实现为教学演示，仅通过head_mask屏蔽被剪枝头的输出（乘以0），
    不减少实际计算量。实际结构化剪枝需移除对应参数并重新构建模型。"""
    def __init__(self, dim=64, n_heads=8):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.head_mask = nn.Parameter(torch.ones(n_heads), requires_grad=False)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        out = out * self.head_mask.view(1, 1, -1).repeat(1, 1, self.head_dim).reshape(1, 1, C)
        return self.proj(out)

    def compute_head_importance(self, dataloader, n_batches=10):
        """计算每个注意力头的重要性分数"""
        head_importance = torch.zeros(self.n_heads)
        self.eval()
        for i, (x, _) in enumerate(dataloader):
            if i >= n_batches:
                break
            self.zero_grad()
            B, N, C = x.shape
            qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
            attn = attn.softmax(dim=-1)
            head_outputs = (attn @ v)
            for h in range(self.n_heads):
                head_importance[h] += head_outputs[:, h].norm().item()
        return head_importance

    def prune_heads(self, heads_to_prune):
        """剪枝指定的注意力头"""
        for h in heads_to_prune:
            self.head_mask[h] = 0.0

dim, n_heads = 64, 8
mha = MultiHeadAttentionWithPruning(dim, n_heads)

x_data = torch.randn(32, 16, dim)
y_data = torch.randint(0, 10, (32,))
dataset = torch.utils.data.TensorDataset(x_data, y_data)
loader = torch.utils.data.DataLoader(dataset, batch_size=8)

head_importance = mha.compute_head_importance(loader)
normalized_importance = head_importance / head_importance.sum()

print(f"=== 注意力头重要性分析 ===")
for h in range(n_heads):
    bar = '█' * int(normalized_importance[h] * 100)
    print(f"Head {h}: {normalized_importance[h]:.4f} {bar}")

sorted_heads = torch.argsort(head_importance)
n_prune = n_heads // 2
heads_to_prune = sorted_heads[:n_prune].tolist()
print(f"\n剪枝头: {heads_to_prune} (重要性最低的{n_prune}个)")

with torch.no_grad():
    out_before = mha(x_data[:4])
    mha.prune_heads(heads_to_prune)
    out_after = mha(x_data[:4])

active_heads = mha.head_mask.sum().item()
print(f"\n剪枝后活跃头数: {int(active_heads)}/{n_heads}")
print(f"注意力计算量减少: {(1 - active_heads/n_heads)*100:.0f}%")
print(f"输出变化: {(out_before - out_after).norm() / out_before.norm() * 100:.2f}%")

### FFN中间维度剪枝 & 层剪枝

#### FFN中间维度剪枝

Transformer的FFN层通常将维度从$d$扩展到$4d$再映射回$d$，中间维度是参数量最大的部分。FFN剪枝识别不重要的中间神经元并移除，直接减少FFN参数量。

重要性评估：基于中间神经元的激活范数
$$I(j) = \sum_{x \in \mathcal{D}} \|h_j(x)\|_1$$

其中 $h_j(x)$ 是第$j$个中间神经元对输入$x$的激活值。

#### 层剪枝

直接移除整个Transformer层。研究表明LLM的相邻层具有高度相似性，移除某些中间层对精度影响较小。层剪枝是最激进的结构化剪枝，每移除一层可减少约$1/N$的总计算量（$N$为总层数）。

In [ ]:
class TransformerFFNWithPruning(nn.Module):
    """支持中间维度剪枝的FFN"""
    def __init__(self, dim=64, intermediate_dim=256):
        super().__init__()
        self.dim = dim
        self.intermediate_dim = intermediate_dim
        self.up_proj = nn.Linear(dim, intermediate_dim)
        self.down_proj = nn.Linear(intermediate_dim, dim)
        self.neuron_mask = nn.Parameter(torch.ones(intermediate_dim), requires_grad=False)

    def forward(self, x):
        h = self.up_proj(x)
        h = F.gelu(h) * self.neuron_mask
        return self.down_proj(h)

    def compute_neuron_importance(self, dataloader, n_batches=10):
        """计算FFN中间神经元的重要性"""
        importance = torch.zeros(self.intermediate_dim)
        self.eval()
        for i, (x, _) in enumerate(dataloader):
            if i >= n_batches:
                break
            with torch.no_grad():
                h = self.up_proj(x)
                h = F.gelu(h)
                importance += h.abs().sum(dim=(0, 1))
        return importance

    def prune_neurons(self, sparsity=0.5):
        """剪枝不重要的中间神经元"""
        importance = self.compute_neuron_importance(loader)
        n_prune = int(sparsity * self.intermediate_dim)
        threshold = importance.kthvalue(n_prune).values
        self.neuron_mask = nn.Parameter((importance > threshold).float(), requires_grad=False)

class TransformerLayerStack(nn.Module):
    """支持层剪枝的Transformer堆叠"""
    def __init__(self, n_layers=6, dim=64, n_heads=4, intermediate_dim=256):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': MultiHeadAttentionWithPruning(dim, n_heads),
                'ffn': TransformerFFNWithPruning(dim, intermediate_dim),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }) for _ in range(n_layers)
        ])
        self.layer_mask = nn.Parameter(torch.ones(n_layers), requires_grad=False)

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            if self.layer_mask[i] == 0:
                continue
            h = layer['ln1'](x)
            h = layer['attn'](h)
            x = x + h
            h = layer['ln2'](x)
            h = layer['ffn'](h)
            x = x + h
        return x

    def compute_layer_importance(self, dataloader, n_batches=5):
        """计算每层的重要性（基于输出变化）"""
        n_layers = len(self.layers)
        importance = torch.zeros(n_layers)
        self.eval()
        for i, (x, _) in enumerate(dataloader):
            if i >= n_batches:
                break
            with torch.no_grad():
                prev = x.clone()
                for l_idx, layer in enumerate(self.layers):
                    h = layer['ln1'](prev)
                    h = layer['attn'](h)
                    out1 = prev + h
                    h = layer['ln2'](out1)
                    h = layer['ffn'](h)
                    out2 = out1 + h
                    importance[l_idx] += (out2 - prev).norm().item()
                    prev = out2
        return importance

ffn = TransformerFFNWithPruning(dim=64, intermediate_dim=256)
ffn.prune_neurons(sparsity=0.5)
active_neurons = ffn.neuron_mask.sum().item()
print(f"=== FFN中间维度剪枝 ===")
print(f"原始中间维度: 256")
print(f"剪枝后活跃维度: {int(active_neurons)}")
print(f"FFN参数量减少: {(1 - active_neurons/256)*100:.0f}%")

stack = TransformerLayerStack(n_layers=6, dim=64, n_heads=4, intermediate_dim=256)
layer_imp = stack.compute_layer_importance(loader)
normalized_imp = layer_imp / layer_imp.sum()

print(f"\n=== 层重要性分析 ===")
for i, imp in enumerate(normalized_imp):
    bar = '█' * int(imp * 100)
    print(f"Layer {i}: {imp:.4f} {bar}")

sorted_layers = torch.argsort(layer_imp)
layers_to_drop = sorted_layers[:2].tolist()
for l in layers_to_drop:
    stack.layer_mask[l] = 0.0

active_layers = stack.layer_mask.sum().item()
print(f"\n剪枝层: {layers_to_drop}")
print(f"活跃层: {int(active_layers)}/6")
print(f"计算量减少: {(1 - active_layers/6)*100:.0f}%")

---
## 1.2.3 半结构化剪枝 / N:M稀疏（Semi-Structured / N:M Sparsity）

### 什么是N:M稀疏？

N:M稀疏是一种半结构化剪枝模式，在每M个连续权重中保留N个非零值（如2:4稀疏表示每4个权重中保留2个），形成固定的稀疏模式。

### 为什么N:M稀疏有效？

1. **硬件原生加速**：NVIDIA Ampere及后续架构的稀疏Tensor Core可直接加速2:4稀疏矩阵运算，获得约2倍加速。这是目前唯一在主流GPU上获得实际加速的稀疏模式。
2. **精度与效率平衡**：50%稀疏度（2:4）下精度损失通常不到1%，远好于非结构化50%稀疏的硬件效率。
3. **确定性稀疏模式**：固定的N:M模式使得硬件可以预先优化内存访问，避免非结构化稀疏的不规则访存问题。

### N:M稀疏的数学公式

对于权重向量$w = [w_1, w_2, ..., w_M]$的每个M块：

$$m_i = \begin{cases} 1 & \text{if } |w_i| \in \text{top-N}(|w_1|, ..., |w_M|) \\ 0 & \text{otherwise} \end{cases}$$

$$w' = w \odot m$$

其中：
- $N$：每M个权重中保留的非零数量
- $M$：分块大小
- $m$：二值掩码
- $\text{top-N}$：取绝对值最大的N个
- 2:4稀疏：50%稀疏度，NVIDIA硬件原生支持
- 1:4稀疏：75%稀疏度，目前尚无主流硬件支持

In [ ]:
def nm_sparsity_prune(weight: torch.Tensor, n: int = 2, m: int = 4) -> tuple:
    """N:M稀疏剪枝：每M个权重中保留N个最大的"""
    assert weight.shape[-1] % m == 0, f"最后一维必须能被{m}整除"
    orig_shape = weight.shape
    w_reshaped = weight.reshape(-1, m)
    abs_vals = w_reshaped.abs()
    topk_vals, topk_idx = abs_vals.topk(n, dim=-1)
    mask = torch.zeros_like(w_reshaped)
    mask.scatter_(-1, topk_idx, 1.0)
    pruned = w_reshaped * mask
    return pruned.reshape(orig_shape), mask.reshape(orig_shape)

def verify_nm_sparsity(mask: torch.Tensor, n: int = 2, m: int = 4):
    """验证N:M稀疏模式是否正确"""
    mask_reshaped = mask.reshape(-1, m)
    counts = mask_reshaped.sum(dim=-1)
    return (counts == n).all().item()

weight = torch.randn(128, 256)

pruned_24, mask_24 = nm_sparsity_prune(weight, n=2, m=4)
pruned_14, mask_14 = nm_sparsity_prune(weight, n=1, m=4)

valid_24 = verify_nm_sparsity(mask_24, n=2, m=4)
valid_14 = verify_nm_sparsity(mask_14, n=1, m=4)

actual_sparsity_24 = 1 - mask_24.mean().item()
actual_sparsity_14 = 1 - mask_14.mean().item()

mse_24 = F.mse_loss(weight, pruned_24)
mse_14 = F.mse_loss(weight, pruned_14)
mse_unstructured_50 = F.mse_loss(weight, magnitude_pruning(weight, 0.5)[0])

print(f"=== N:M稀疏剪枝 ===")
print(f"2:4稀疏 (50%稀疏度):")
print(f"  验证通过: {valid_24}")
print(f"  实际稀疏度: {actual_sparsity_24:.2%}")
print(f"  MSE: {mse_24:.6f}")
print(f"\n1:4稀疏 (75%稀疏度):")
print(f"  验证通过: {valid_14}")
print(f"  实际稀疏度: {actual_sparsity_14:.2%}")
print(f"  MSE: {mse_14:.6f}")
print(f"\n对比 - 非结构化50%稀疏MSE: {mse_unstructured_50:.6f}")
print(f"2:4稀疏MSE: {mse_24:.6f}")
print(f"\n关键: 2:4稀疏有硬件加速支持(NVIDIA稀疏Tensor Core)，实际推理可获得~2x加速")

### N:M稀疏的稀疏感知训练恢复精度

#### 什么是稀疏感知训练？

稀疏感知训练是在N:M稀疏约束下微调模型，让剩余的非零权重调整以补偿被剪枝权重的影响。每次梯度更新后，强制将权重投影到N:M稀疏模式上。

#### 为什么稀疏感知训练能恢复精度？

1. **权重补偿**：剪枝后剩余的权重通过训练调整，学习补偿被移除权重的功能。
2. **稀疏约束下的最优解**：训练过程找到在N:M稀疏约束下的最优权重配置，而非先训练后剪枝的次优解。
3. **实践效果**：2:4稀疏+稀疏感知训练通常可恢复95%以上的精度损失。

In [ ]:
def sparse_aware_training(model, train_loader, n=2, m=4, epochs=20, lr=1e-3):
    """稀疏感知训练：在N:M稀疏约束下微调模型"""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_history = []

    for epoch in range(epochs):
        epoch_loss = 0
        n_batches = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            output = model(x)
            loss = F.cross_entropy(output, y)
            loss.backward()
            optimizer.step()

            for name, param in model.named_parameters():
                if 'weight' in name and param.dim() >= 2:
                    _, mask = nm_sparsity_prune(param.data, n=n, m=m)
                    param.data *= mask

            epoch_loss += loss.item()
            n_batches += 1
        loss_history.append(epoch_loss / n_batches)

    return loss_history

model_sparse = SimpleMLP(dim=64)

with torch.no_grad():
    out_dense = model_sparse(x_data)
    acc_dense = (out_dense.argmax(1) == y_data).float().mean()

for name, param in model_sparse.named_parameters():
    if 'weight' in name and param.dim() >= 2:
        _, mask = nm_sparsity_prune(param.data, n=2, m=4)
        param.data *= mask

with torch.no_grad():
    out_pruned = model_sparse(x_data)
    acc_pruned = (out_pruned.argmax(1) == y_data).float().mean()

loss_hist = sparse_aware_training(model_sparse, loader, n=2, m=4, epochs=15)

with torch.no_grad():
    out_finetuned = model_sparse(x_data)
    acc_finetuned = (out_finetuned.argmax(1) == y_data).float().mean()

total_params = sum(p.numel() for p in model_sparse.parameters())
zero_params = sum((p == 0).sum().item() for p in model_sparse.parameters())

print(f"=== 2:4稀疏感知训练 ===")
print(f"稠密模型准确率: {acc_dense:.4f}")
print(f"剪枝后准确率: {acc_pruned:.4f}")
print(f"微调后准确率: {acc_finetuned:.4f}")
print(f"实际稀疏度: {zero_params/total_params:.2%}")
print(f"训练损失: {loss_hist[0]:.4f} -> {loss_hist[-1]:.4f}")
print(f"\n结论: 稀疏感知训练可显著恢复剪枝带来的精度损失")

### 综合对比：不同剪枝方法效果

不同剪枝方法在稀疏度、硬件加速和精度损失方面的对比。选择剪枝方法时需综合考虑目标硬件和精度要求。

In [ ]:
print(f"{'方法':<25} {'稀疏度':<10} {'硬件加速':<15} {'精度损失':<15}")
print("-" * 65)
methods = [
    ("幅度剪枝(非结构化)", "任意", "需要稀疏支持", "最小"),
    ("梯度剪枝(非结构化)", "任意", "需要稀疏支持", "较小"),
    ("GMP(渐进式)", "任意", "需要稀疏支持", "比一次性更小"),
    ("注意力头剪枝(结构化)", "按头", "直接加速", "中等"),
    ("FFN维度剪枝(结构化)", "按维度", "直接加速", "中等"),
    ("层剪枝(结构化)", "按层", "直接加速", "可能较大"),
    ("2:4稀疏(半结构化)", "50%", "NVIDIA加速2x", "小+可微调恢复"),
    ("1:4稀疏(半结构化)", "75%", "需硬件支持", "较大"),
]
for name, sp, hw, acc in methods:
    print(f"{name:<25} {sp:<10} {hw:<15} {acc:<15}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. NVIDIA GPU端侧: 2:4稀疏 + 稀疏感知训练 (硬件原生支持)")
print(f"2. CPU/NPU端侧: 结构化剪枝 (无需稀疏硬件支持)")
print(f"3. 极致压缩: 结构化剪枝 + 量化 + 蒸馏联合")
print(f"4. 精度优先: 非结构化剪枝 + GMP渐进训练")

### 产业级实战：真实结构化剪枝（FFN中间维度裁剪）

产业界的结构化剪枝不是用mask屏蔽，而是真正移除参数并重建更小的模型。以下展示FFN中间维度剪枝的完整流程：评估重要性 → 裁剪 → 重建小模型 → 验证精度。

**关键步骤**：
1. **校准**：用代表性数据评估各神经元/头/层的重要性（基于激活范数或梯度信息）
2. **裁剪**：真正移除参数，创建更小的权重矩阵，重建更小的模型
3. **微调**：在训练数据上微调恢复精度（关键步骤，通常需要原训练数据的子集）
4. **迭代**：可多次迭代"剪枝→微调"逐步压缩，每次小幅剪枝后微调

In [ ]:
import time
import copy

class PrunableFFN(nn.Module):
    def __init__(self, dim, ffn_dim):
        super().__init__()
        self.up_proj = nn.Linear(dim, ffn_dim)
        self.down_proj = nn.Linear(ffn_dim, dim)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x)))

def evaluate_neuron_importance(ffn_layer, calib_data, top_k_ratio=0.5):
    """评估FFN中间神经元的重要性（基于激活范数）"""
    activations = []
    hooks = []
    def hook_fn(module, input, output):
        activations.append(output.detach())
    hooks.append(ffn_layer.up_proj.register_forward_hook(hook_fn))

    with torch.no_grad():
        for batch in calib_data:
            ffn_layer(batch)

    for h in hooks:
        h.remove()

    all_acts = torch.cat(activations, dim=0)
    importance = all_acts.abs().mean(dim=(0, 1))
    k = max(1, int(len(importance) * top_k_ratio))
    topk_indices = torch.topk(importance, k).indices.sort()[0]
    return topk_indices, importance

def prune_ffn_dimension(ffn_layer, keep_indices):
    """真正裁剪FFN中间维度，创建更小的模型"""
    new_ffn_dim = len(keep_indices)
    dim = ffn_layer.up_proj.in_features

    new_ffn = PrunableFFN(dim, new_ffn_dim)
    with torch.no_grad():
        new_ffn.up_proj.weight.copy_(ffn_layer.up_proj.weight[keep_indices])
        if ffn_layer.up_proj.bias is not None:
            new_ffn.up_proj.bias.copy_(ffn_layer.up_proj.bias[keep_indices])
        new_ffn.down_proj.weight.copy_(ffn_layer.down_proj.weight[:, keep_indices])
        if ffn_layer.down_proj.bias is not None:
            new_ffn.down_proj.bias.copy_(ffn_layer.down_proj.bias)
    return new_ffn

dim, ffn_dim = 128, 512
ffn = PrunableFFN(dim, ffn_dim)
ffn.eval()

calib_data = [torch.randn(8, dim) for _ in range(10)]
test_input = torch.randn(16, dim)

with torch.no_grad():
    original_output = ffn(test_input)

keep_indices, importance = evaluate_neuron_importance(ffn, calib_data, top_k_ratio=0.5)
pruned_ffn = prune_ffn_dimension(ffn, keep_indices)

with torch.no_grad():
    pruned_output = pruned_ffn(test_input)

original_params = sum(p.numel() for p in ffn.parameters())
pruned_params = sum(p.numel() for p in pruned_ffn.parameters())
original_mem = sum(p.numel() * p.element_size() for p in ffn.parameters()) / 1024
pruned_mem = sum(p.numel() * p.element_size() for p in pruned_ffn.parameters()) / 1024

start = time.perf_counter()
with torch.no_grad():
    for _ in range(100):
        ffn(test_input)
original_time = (time.perf_counter() - start) / 100 * 1000

start = time.perf_counter()
with torch.no_grad():
    for _ in range(100):
        pruned_ffn(test_input)
pruned_time = (time.perf_counter() - start) / 100 * 1000

print(f"=== 真实结构化剪枝: FFN中间维度裁剪 ===")
print(f"原始FFN: {dim} → {ffn_dim} → {dim}")
print(f"剪枝后:  {dim} → {len(keep_indices)} → {dim}")
print(f"\n{'指标':<20} {'原始':<15} {'剪枝后':<15} {'变化':<15}")
print("-" * 65)
print(f"{'参数量':<20} {original_params:<15,} {pruned_params:<15,} {(1-pruned_params/original_params)*100:.0f}%↓")
print(f"{'内存(KB)':<20} {original_mem:<15.1f} {pruned_mem:<15.1f} {(1-pruned_mem/original_mem)*100:.0f}%↓")
print(f"{'延迟(ms)':<20} {original_time:<15.2f} {pruned_time:<15.2f} {(1-pruned_time/original_time)*100:.0f}%↓")

print(f"\n产业界结构化剪枝流程:")
print(f"1. 校准: 用代表性数据评估各神经元/头/层的重要性")
print(f"2. 裁剪: 真正移除参数，重建更小的模型")
print(f"3. 微调: 在训练数据上微调恢复精度 (关键步骤)")
print(f"4. 迭代: 可多次迭代'剪枝→微调'逐步压缩")
print(f"5. 工具: torch.nn.utils.prune (非结构化) / 自定义裁剪 (结构化)")